# Prediction of Product Sales

- Author: Salam

## Load and Inspect Data

In [ ]:
# Mount Google Drive (Colab specific - skip if running locally)

from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Import libraries

import pandas as pd

import numpy as np

pd.set_option('display.max_columns', 100)

In [ ]:
# Load the sales prediction dataset using Pandas

# Adjust this path to wherever you saved the CSV in your Drive

fpath = '/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Project1/Data/sales_predictions_2023.csv'



df = pd.read_csv(fpath)

df.head()

In [ ]:
# Use .info() to preview datatypes and a summary of the DataFrame's columns

df.info()

### 1) How many rows and columns?

In [ ]:
# .shape returns (n_rows, n_columns)

n_rows, n_cols = df.shape

print(f"Rows: {n_rows}")

print(f"Columns: {n_cols}")

- The dataset has **8,523 rows** and **12 columns**.

### 2) What are the datatypes of each variable?

In [ ]:
# .dtypes lists the datatype of every column

df.dtypes

- Numeric columns (`float64`/`int64`): `Item_Weight`, `Item_Visibility`, `Item_MRP`, `Outlet_Establishment_Year`, `Item_Outlet_Sales` (target)

- Categorical columns (`object`): `Item_Identifier`, `Item_Fat_Content`, `Item_Type`, `Outlet_Identifier`, `Outlet_Size`, `Outlet_Location_Type`, `Outlet_Type`

### 3) Are there duplicates? If so, drop any duplicates.

In [ ]:
# Check for fully duplicated rows

print(f"Number of duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Drop duplicates (if any). Even if 0, it is good practice to include this step.

df = df.drop_duplicates()



# Confirm shape after dropping duplicates

print(f"Shape after dropping duplicates: {df.shape}")

- There were **0 duplicate rows** in this dataset, so no rows were removed.

### 4) Identify missing values.

In [ ]:
# Check for missing values in each column

df.isna().sum()

- `Item_Weight`: 1,463 missing values (numeric column)

- `Outlet_Size`: 2,410 missing values (categorical column)

- All other columns have 0 missing values.

### 5) Address the missing values by using a placeholder value.

In [ ]:
# --- Item_Weight (numeric) ---

# Fill missing numeric values with a placeholder of 0

# (Note: for modeling later, we will use a proper SimpleImputer with

#  median strategy AFTER the train-test split to avoid data leakage.

#  For this initial EDA/cleaning pass, we use a simple placeholder

#  so our null-checking functions can report on this column.)

df['Item_Weight'] = df['Item_Weight'].fillna(0)



# --- Outlet_Size (categorical) ---

# Fill missing categorical values with the placeholder string 'MISSING'

df['Outlet_Size'] = df['Outlet_Size'].fillna('MISSING')



# Preview the changes

df[['Item_Weight', 'Outlet_Size']].isna().sum()

### 6) Confirm that there are no missing values after addressing them.

In [ ]:
# Confirm there are no remaining missing values anywhere in the dataframe

df.isna().sum()

- All columns now show **0 missing values**.

### 7) Find and fix any inconsistent categories of data.

In [ ]:
# Check each categorical column's unique values for inconsistencies

cat_cols = df.select_dtypes('object').columns



for col in cat_cols:

    print(f"--- {col} ---")

    print(df[col].unique())

    print()

- `Item_Fat_Content` has inconsistent categories representing the same two groups:

  - `'Low Fat'`, `'low fat'`, `'LF'` should all be **`'Low Fat'`**

  - `'Regular'`, `'reg'` should all be **`'Regular'`**

- All other categorical columns look consistent.

In [ ]:
# Check value counts BEFORE fixing

df['Item_Fat_Content'].value_counts()

In [ ]:
# Fix inconsistent categories using a mapping dictionary with .replace()

fat_content_map = {

    'low fat': 'Low Fat',

    'LF': 'Low Fat',

    'reg': 'Regular'

}



df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_content_map)



# Check value counts AFTER fixing - should now only be 2 categories

df['Item_Fat_Content'].value_counts()

### 8) For any numerical columns, obtain the summary statistics of each (min, max, mean).

In [ ]:
# .describe() returns summary statistics (count, mean, std, min, 25%, 50%, 75%, max)

# for all numeric columns

df.describe()

In [ ]:
# Focus on just min, max, and mean for a quick summary

df.describe().loc[['min', 'max', 'mean']]

**Summary statistics observations:**



- `Item_Weight`: ranges from 0 (placeholder for missing) to ~21.35, mean ~11.85

- `Item_Visibility`: ranges from 0 to ~0.33, mean ~0.066 (the percentage of total display area)

- `Item_MRP`: ranges from ~31.29 to ~266.89, mean ~140.99 (price of the item)

- `Outlet_Establishment_Year`: ranges from 1985 to 2009

- `Item_Outlet_Sales` (target): ranges from ~33.29 to ~13,086.96, mean ~2,181.29

## Data Dictionary



| Variable Name | Description |

|---|---|

| Item_Identifier | Unique product ID |

| Item_Weight | Weight of product |

| Item_Fat_Content | Whether the product is low fat or regular |

| Item_Visibility | The percentage of total display area of all products in a store allocated to the particular product |

| Item_Type | The category to which the product belongs |

| Item_MRP | Maximum Retail Price (list price) of the product |

| Outlet_Identifier | Unique store ID |

| Outlet_Establishment_Year | The year in which store was established |

| Outlet_Size | The size of the store in terms of ground area covered |

| Outlet_Location_Type | The type of area in which the store is located |

| Outlet_Type | Whether the outlet is a grocery store or some sort of supermarket |

| Item_Outlet_Sales | Sales of the product in the particular store. This is the **target variable** to be predicted. |

## Clean Data



Summary of cleaning performed in this notebook:

1. Checked for and confirmed no duplicate rows.

2. Identified missing values in `Item_Weight` (1,463) and `Outlet_Size` (2,410).

3. Addressed missing values with placeholders (`0` for `Item_Weight`, `'MISSING'` for `Outlet_Size`) so EDA functions can report on null presence/frequency.

4. Fixed inconsistent categories in `Item_Fat_Content` (`low fat`/`LF` -> `Low Fat`, `reg` -> `Regular`).

5. Reviewed summary statistics (min/max/mean) for all numeric columns.



**Note:** In Part 5, we will reload the ORIGINAL (uncleaned) dataset from scratch and perform the train/test split BEFORE imputing missing values (using `SimpleImputer`), to avoid data leakage. The placeholder-based cleaning above is for this week's exploratory work only.